# Predictive Component

Trains a Random Forest to estimate, for each canton and El Nino episode, the people, crop area and houses affected and the probability of a severe outcome. Validation is leave-one-episode-out and always compared against a simple baseline (the canton's historical mean).

Outputs: metrics, predictions, feature importances and the trained model.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Numerical operations
import numpy as np
# Tabular data handling
import pandas as pd

# Make the project's src package importable
sys.path.insert(0, str(Path.cwd().parent))
# Predictive-component functions
from src.analisis import prediccion as pr
# Helper to load the YAML config
from src.utilidades.config_utils import load_config
# Project logger factory
from src.utilidades.logging_utils import get_logger

# Load the configuration
config = load_config(Path.cwd().parent / "config" / "config.yaml")
# Create a logger that writes to the logs folder
logger = get_logger("step12_predictive", config, log_dir=Path.cwd().parent / "logs")
# Log that the notebook started
logger.info("componente predictivo: componente predictivo iniciado")

## 1. Load data

In [ ]:
# Project root
base_dir = Path.cwd().parent
# Load the hazard events
hazard_event = pd.read_parquet(base_dir / "data/processed/hazard_event.parquet")
# Load the extended monthly ENSO series
enso_monthly = pd.read_parquet(base_dir / "data/processed/enso_index_monthly_extended.parquet")
# Load the canton recurrence table
recurrencia = pd.read_csv(base_dir / "data/processed/recurrence_index_canton.csv")
# Universe of the 221 cantons
universo_cantones = recurrencia.loc[recurrencia["hazard_type"] == "TODOS", ["canton_code"]].drop_duplicates()

# Log the number of events
logger.info("hazard_event: %d filas", len(hazard_event))
# Log the number of months
logger.info("enso_monthly: %d meses", len(enso_monthly))
# Log the number of cantons
logger.info("universo de cantones: %d", len(universo_cantones))
# Check there are exactly 221 cantons
assert len(universo_cantones) == 221, "El universo de cantones debe tener 221 filas"

## 2. Definition and verification of ENSO episodes

In [ ]:
# Set of (year, month) pairs flagged as El Nino in the ENSO series
eventos_enso_reales = set(
    zip(
        enso_monthly.loc[enso_monthly["es_evento_enso"], "year"],
        enso_monthly.loc[enso_monthly["es_evento_enso"], "month"],
    )
)
# Union of the months of all defined episodes
union_episodios = set()
# Loop over the episodes
for meses in pr.EPISODIOS_ENSO.values():
    # Add the months of each episode
    union_episodios |= set(meses)

# Check the episodes cover exactly the El Nino months of the ENSO series
assert union_episodios == eventos_enso_reales, (
    f"Los episodios definidos no reproducen exactamente los meses El Niño reales. "
    f"Faltantes: {eventos_enso_reales - union_episodios}, sobrantes: {union_episodios - eventos_enso_reales}"
)
# Log that the episodes match the El Nino months
logger.info("Verificado: los 6 episodios definidos cubren exactamente los 38 meses El Niño de recurrencia")

# Loop over the episodes
for nombre, meses in pr.EPISODIOS_ENSO.items():
    # Months of the episode as a set
    meses_set = set(meses)
    # Flag events that fall in the episode
    mask = hazard_event.apply(lambda r: (int(r["year"]), int(r["month"])) in meses_set, axis=1)
    # Number of events in the episode
    n = int(mask.sum())
    # Log the number of months and events of the episode
    logger.info("Episodio %-10s: %3d meses, %5d eventos de peligro reales", nombre, len(meses), n)
    # Check the episode has events, so it can serve as a validation fold
    assert n > 0, f"Episodio {nombre} no tiene ningún evento — no se puede usar como fold"

## 3. Canton x episode panel

In [ ]:
# Build the canton x episode panel of impacts
panel = pr.construir_panel_canton_episodio(hazard_event, enso_monthly, universo_cantones)

# Check the panel has 221 cantons x 6 episodes
assert len(panel) == 221 * 6, "El panel debe tener exactamente 221 cantones x 6 episodios"
# Log the number of panel rows
logger.info("Panel construido: %d filas", len(panel))

# Total impacts per episode
resumen_episodio = panel.groupby("episodio")[pr.TARGET_COLS + ["n_eventos"]].sum()
# Add the mean ENSO magnitude of each episode
resumen_episodio["magnitud_enso_media"] = panel.groupby("episodio")["magnitud_enso_media"].first()
# Show the episode summary
resumen_episodio

## 4. Binary severity label

In [ ]:
# Binary severity label: top 20% of the impact distribution
panel["severo"] = pr.severidad_binaria(panel, umbral_percentil=0.80)
# Number of severe rows
n_severos = int(panel["severo"].sum())
# Log the share of severe rows
logger.info("Filas marcadas como 'severo': %d de %d (%.1f%%)", n_severos, len(panel), 100 * n_severos / len(panel))
# Show severe rows per episode
panel.groupby("episodio")["severo"].sum()

## 5. Leave-one-episode-out validation (model versus baseline)

In [ ]:
# Leave-one-episode-out validation of the model against the baseline
resultado = pr.evaluar_loeo(panel, random_state=42, n_estimators=300)
# Metrics per fold
metricas = resultado["metricas_fold"]
# Individual predictions
predicciones = resultado["predicciones"]
# Feature importances
importancia = resultado["importancia_features"]

# Log the size of the results
logger.info("LOEO completo: %d filas de métricas, %d predicciones individuales", len(metricas), len(predicciones))
# Show the fold metrics
metricas

## 6. Aggregated results: MAE (regression)

In [ ]:
# Select the mean absolute error rows
mae = metricas[metricas["metrica"] == "MAE"].copy()
# Mark folds where the model beats the baseline
mae["modelo_mejor"] = mae["modelo"] < mae["baseline"]
# Percentage error reduction of the model versus the baseline
mae["reduccion_error_pct"] = 100 * (1 - mae["modelo"] / mae["baseline"])

# Log the coverage check
logger.info(
    "MAE: el modelo supera al baseline en %d de %d combinaciones episodio x target",
    int(mae["modelo_mejor"].sum()), len(mae),
)
# Log the mean error reduction
logger.info("Reducción media de error (MAE) del modelo vs. baseline: %.1f%%", mae["reduccion_error_pct"].mean())
# Show MAE by episode and target
mae.pivot_table(index="episodio", columns="target", values=["modelo", "baseline"])

## 7. Aggregated results: severity classification (AUC, F1)

In [ ]:
# Select the AUC rows without missing values
auc = metricas[metricas["metrica"] == "AUC"].dropna()
# Select the F1 rows
f1 = metricas[metricas["metrica"] == "F1"]

# Log the mean AUC of model and baseline
logger.info("AUC promedio — modelo: %.3f, baseline: %.3f (mejor en %d/%d episodios)",
            auc["modelo"].mean(), auc["baseline"].mean(), int((auc["modelo"] > auc["baseline"]).sum()), len(auc))
# Log the mean F1 of model and baseline
logger.info("F1 promedio — modelo: %.3f, baseline: %.3f (mejor en %d/%d episodios)",
            f1["modelo"].mean(), f1["baseline"].mean(), int((f1["modelo"] > f1["baseline"]).sum()), len(f1))

# Show severity metrics by episode
metricas[metricas["target"] == "severo"].pivot_table(index="episodio", columns="metrica", values=["modelo", "baseline"])

## 8. R-squared: reported for transparency, not as the main metric

In [ ]:
# Select the R-squared rows
r2 = metricas[metricas["metrica"] == "R2"]
# Log the mean R-squared
logger.info("R2 promedio — modelo: %.2f, baseline: %.2f", r2["modelo"].mean(), r2["baseline"].mean())
# Show R-squared by episode and target
r2.pivot_table(index="episodio", columns="target", values=["modelo", "baseline"])

## 9. Feature importance (final model, trained on all episodes)

In [ ]:
# Sort features by importance
importancia_ordenada = importancia.set_index("feature").sort_values("vivienda_afectada", ascending=False)
# Show the feature importances
importancia_ordenada

## 10. Export

In [ ]:
# Model serialization library
import joblib

# Output folder
out_dir = base_dir / "data" / "processed"
# Save the panel as CSV
panel.to_csv(out_dir / "prediccion_panel_canton_episodio.csv", index=False)
# Save the panel as Parquet
panel.to_parquet(out_dir / "prediccion_panel_canton_episodio.parquet", index=False)
# Save the fold metrics
metricas.to_csv(out_dir / "prediccion_metricas_loeo.csv", index=False)
# Save the individual predictions
predicciones.to_csv(out_dir / "prediccion_predicciones_loeo.csv", index=False)
# Save the feature importances
importancia.to_csv(out_dir / "prediccion_importancia_features.csv", index=False)

# Feature columns used by the final model
feature_cols = [
    "magnitud_enso_media", "magnitud_enso_max", "hist_n_episodios_con_evento",
    "hist_frac_episodios_con_evento", "hist_poblacion_media", "hist_agricultura_media",
    "hist_vivienda_media", "hist_n_eventos_medio",
]
# Build historical features from all episodes, without leakage
hist_final = pr.construir_features_sin_fuga(panel.assign(episodio="__ninguno__"), "__no_excluir_nada__")
# Add the features to the panel
panel_final = panel.merge(hist_final, on="canton_code", how="left")
# Replace missing features with zero
panel_final[feature_cols] = panel_final[feature_cols].fillna(0.0)

# Random Forest regressor and classifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Folder for the trained models
modelos_dir = base_dir / "models"
# Create the folder if needed
modelos_dir.mkdir(exist_ok=True)
# Dictionary that collects the final models
modelos_finales = {}
# Loop over the impact targets
for col in pr.TARGET_COLS:
    # Random Forest regressor for the target
    m = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=3)
    # Fit on the log-transformed target
    m.fit(panel_final[feature_cols], np.log1p(panel_final[col]))
    # Store the regressor
    modelos_finales[f"regresor_{col}"] = m

# Random Forest classifier for the severe label, with balanced class weights
clf = RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=3, class_weight="balanced")
# Fit the classifier
clf.fit(panel_final[feature_cols], panel_final["severo"].astype(int))
# Store the classifier
modelos_finales["clasificador_severo"] = clf

# Save all models and the feature list
joblib.dump({"modelos": modelos_finales, "feature_cols": feature_cols}, modelos_dir / "step12_modelos_finales.joblib")

# Log that all outputs were saved
logger.info("Exportación completa: panel, métricas, predicciones, importancia y modelos finales guardados")
# Log that the run is complete
logger.info("componente predictivo (componente predictivo) — ejecución completa")